In [3]:
import cirq
import numpy as np

def calcular_cerebro_cuantico_28q_cirq(thetas: list, phases: list, phis: list, shots: int = 100):
    """
    Simulación del Cerebro Cuántico de 28 Qubits usando Google Cirq.

    Aplica rotaciones Ry(θ) y Rz(λ) seguidas de una estructura de
    Entrelazamiento Jerárquico Circular (HEC) en 7 bloques con compuertas CRY(φ).
    """
    # 1. Definición de 28 Qubits en línea en la infraestructura de Cirq
    qubits = cirq.LineQubit.range(28)
    circuit = cirq.Circuit()

    # 2. Aplicar rotaciones de Amplitud Ry(θ) y Fase Rz(λ)
    for i in range(28):
        circuit.append(cirq.ry(thetas[i])(qubits[i]))
        circuit.append(cirq.rz(phases[i])(qubits[i]))

    # Función auxiliar para compuertas Controlled-Ry (CRY) en Cirq
    def aplicar_cry(phi_val, ctrl_idx, tgt_idx):
        cry_gate = cirq.ControlledGate(sub_gate=cirq.ry(phi_val))
        circuit.append(cry_gate(qubits[ctrl_idx], qubits[tgt_idx]))

    # 3. Estructura de Entrelazamiento Jerárquico en Bloques de 4 Qubits (HEC - 7 Bloques)
    # B1 -> B2 (q0-q3 entrelazados con q4-q7)
    for i in range(4):
        aplicar_cry(phis[0], i, i + 4)

    # B2 -> B3 (q4-q7 entrelazados con q8-q11)
    for i in range(4, 8):
        aplicar_cry(phis[1], i, i + 4)

    # B3 -> B4 (q8-q11 entrelazados con q12-q15)
    for i in range(8, 12):
        aplicar_cry(phis[2], i, i + 4)

    # B4 -> B5 (q12-q15 entrelazados con q16-q19)
    for i in range(12, 16):
        aplicar_cry(phis[3], i, i + 4)

    # B5 -> B6 (q16-q19 entrelazados con q20-q23)
    for i in range(16, 20):
        aplicar_cry(phis[0], i, i + 4)

    # B6 -> B7 (q20-q23 entrelazados con q24-q27)
    for i in range(20, 24):
        aplicar_cry(phis[1], i, i + 4)

    # Cierre de Anillo de Propósito Circular (q27 entrelazado con q0)
    aplicar_cry(phis[2], 27, 0)

    # 4. Medición de todos los qubits en el simulador de Cirq
    circuit.append(cirq.measure(*qubits, key='result'))

    # 5. Ejecución en el simulador cuasi-físico de Google Cirq
    simulator = cirq.Simulator()
    result = simulator.run(circuit, repetitions=shots)

    # Matriz binaria de mediciones: forma (shots, 28)
    measurements = result.measurements['result']

    # 6. Cálculo de Probabilidades Marginales de cada Qubit P(q_i = 1)
    marginal_probs = np.mean(measurements, axis=0).tolist()

    # 7. Cálculo del Índice de Caos (Entropía Cuántica de Shannon)
    unique_states, counts = np.unique(measurements, axis=0, return_counts=True)
    probs_states = counts / shots
    entropia = -np.sum(probs_states * np.log2(probs_states))
    max_entropia = np.log2(min(shots, 2**28))
    indice_caos = float((entropia / max_entropia) * 100.0)

    # 8. Determinación del Canal Cognitivo PNL dominante (q0-q3)
    pnl_scores = marginal_probs[0:4]
    canales = ["KINESTÉSICO", "VISUAL", "AUDITIVO", "DIGITAL"]
    canal_dominante = canales[int(np.argmax(pnl_scores))]

    return {
        "marginal_probs": marginal_probs,
        "indice_caos": indice_caos,
        "canal_pnl": canal_dominante,
        "circuit_depth": len(circuit)
    }

# Ejemplo de prueba de ejecución
if __name__ == "__main__":
    thetas_init = [np.pi / 4] * 28
    phases_init = [0.0] * 28
    phis_init = [np.pi / 6] * 4

    res = calcular_cerebro_cuantico_28q_cirq(thetas_init, phases_init, phis_init)
    print(f"✅ Circuito Cirq Ejecutado | Profundidad: {res['circuit_depth']} capas")
    print(f"📊 Canal PNL Dominante: {res['canal_pnl']}")
    print(f"🌀 Índice de Caos (Entropía): {res['indice_caos']:.2f}%")

✅ Circuito Cirq Ejecutado | Profundidad: 10 capas
📊 Canal PNL Dominante: VISUAL
🌀 Índice de Caos (Entropía): 100.00%


In [4]:
!pip install -q cirq google-genai pydantic numpy

In [5]:
import os
import json
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# Inserte su GEMINI_API_KEY o use Google Colab Secrets (userdata)
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY') if 'GEMINI_API_KEY' in os.environ else "TU_GEMINI_API_KEY"

client = genai.Client(api_key=api_key)
GEMINI_MODEL = "gemini-3.1-flash-lite"

class AnalisisCognitivo(BaseModel):
    pnl_kinestesico: float = Field(description="Nivel de lenguaje de acción/movimiento (0.0 a 1.0)")
    pnl_visual: float = Field(description="Nivel de lenguaje conceptual/imágenes (0.0 a 1.0)")
    pnl_auditivo: float = Field(description="Nivel de lenguaje de diálogo/escucha (0.0 a 1.0)")
    pnl_digital: float = Field(description="Nivel de lenguaje lógico/datos (0.0 a 1.0)")
    burnout: float = Field(description="Nivel de estrés/saturación (0.0 a 1.0)")
    resilience: float = Field(description="Nivel de proactividad/adaptabilidad (0.0 a 1.0)")
    sentiment_score: float = Field(description="Sentimiento general (-1.0 a 1.0)")

def extraer_pnl(texto: str):
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=f"Analiza la siguiente frase: '{texto}'",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=AnalisisCognitivo,
            temperature=0.1
        )
    )
    return json.loads(response.text)

In [6]:
# Frase de prueba con alto nivel de estrés/burnout
texto_usuario = "Estoy totalmente desbordado con la gestión de los presupuestos y el equipo está agotado de tantas reuniones."

print("1. Analizando estado cognitivo con Gemini API...")
analisis = extraer_pnl(texto_usuario)
print("   Resultado PNL/Estrés:", json.dumps(analisis, indent=2))

# Mapear valores a parámetros del circuito cuántico de Cirq
thetas = [np.pi/4] * 28
phases = [0.0] * 28
phis = [np.pi/6] * 4

# Modular amplitudes con la salida de Gemini
thetas[0] = analisis["pnl_kinestesico"] * np.pi
thetas[1] = analisis["pnl_visual"] * np.pi
thetas[2] = analisis["pnl_auditivo"] * np.pi
thetas[3] = analisis["pnl_digital"] * np.pi
thetas[5] = analisis["burnout"] * np.pi # q5: Burnout

print("\n2. Simulando estado en el motor cuántico Google Cirq (28 Qubits)...")
telemetria = calcular_cerebro_cuantico_28q_cirq(thetas, phases, phis)

print(f"   - Canal PNL Reconstruido: {telemetria['canal_pnl']}")
print(f"   - Índice de Caos Sistémico: {telemetria['indice_caos']:.2f}%")
print(f"   - Probabilidad Marginal de Burnout (q5): {telemetria['marginal_probs'][5]*100:.1f}%")

1. Analizando estado cognitivo con Gemini API...


ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}